In [1]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

In [17]:
df = pd.read_csv('C:/Users/peili/Downloads/hts_media_campaigns.csv')


In [18]:
today = pd.to_datetime("2026-03-01")

df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])

# Time calculations
df["days_total"] = (df["end_date"] - df["start_date"]).dt.days
df["days_elapsed"] = (today - df["start_date"]).dt.days
df["days_remaining"] = (df["end_date"] - today).dt.days

df["time_elapsed_pct"] = df["days_elapsed"] / df["days_total"]
df["spend_pct"] = df["spend_to_date"] / df["budget_total"]

# Pacing
df["pacing_ratio"] = df["spend_pct"] / df["time_elapsed_pct"]

# Required daily spend
df["budget_remaining"] = df["budget_total"] - df["spend_to_date"]
df["required_daily_spend"] = df["budget_remaining"] / df["days_remaining"]

df["required_vs_actual_ratio"] = df["required_daily_spend"] / df["avg_daily_spend_7d"]

In [15]:
print(df[["campaign_id", "advertiser", "time_elapsed_pct", "spend_pct", "pacing_ratio", "required_daily_spend", "required_vs_actual_ratio"]])
df.columns

   campaign_id              advertiser  time_elapsed_pct  spend_pct  \
0           C1     Vegas Tourism Board          0.482759   0.350000   
1           C2         Orlando Tourism          0.500000   0.277778   
2           C3        NYC Hotels Group          0.762712   0.733333   
3           C4          Hawaii Resorts          0.130435   0.187500   
4           C5              DC Tourism          0.795455   0.633333   
5           C6      LA Boutique Hotels          0.500000   0.360000   
6           C7       Miami Beach Board          0.325581   0.171429   
7           C8     Chicago City Travel          0.925926   0.900000   
8           C9   Seattle Luxury Hotels          0.383562   0.444444   
9          C10         Phoenix Resorts          0.666667   0.373333   
10         C11        San Diego Travel          0.677966   0.684211   
11         C12     Boston Hotels Assoc          0.678571   0.254545   
12         C13          Denver Tourism          0.909091   0.769231   
13    

Index(['campaign_id', 'advertiser', 'budget_total', 'spend_to_date',
       'start_date', 'end_date', 'daily_cap', 'avg_daily_spend_7d',
       'ctr_trend', 'bid_level', 'notes_main', 'status_signal',
       'creative_issue', 'tracking_issue', 'bid_issue', 'legal_delay',
       'placement_issue', 'days_total', 'days_elapsed', 'days_remaining',
       'time_elapsed_pct', 'spend_pct', 'pacing_ratio', 'budget_remaining',
       'required_daily_spend', 'required_vs_actual_ratio'],
      dtype='object')

In [19]:
df["performance_decline"] = df["notes_main"].str.contains("declin", case=False).astype(int)
df["premium_partner"] = df["notes_main"].str.contains("premium", case=False).astype(int)
df["over_pacing_flag"] = df["notes_main"].str.contains("over pacing", case=False).astype(int)
df["learning_phase"] = df["notes_main"].str.contains("learning", case=False).astype(int)

In [24]:
print (df[["advertiser", "pacing_ratio", "performance_decline", "premium_partner", "over_pacing_flag", "learning_phase"]])

                advertiser  pacing_ratio  performance_decline  \
0      Vegas Tourism Board      0.725000                    0   
1          Orlando Tourism      0.555556                    0   
2         NYC Hotels Group      0.961481                    0   
3           Hawaii Resorts      1.437500                    0   
4               DC Tourism      0.796190                    0   
5       LA Boutique Hotels      0.720000                    0   
6        Miami Beach Board      0.526531                    0   
7      Chicago City Travel      0.972000                    0   
8    Seattle Luxury Hotels      1.158730                    0   
9          Phoenix Resorts      0.560000                    0   
10        San Diego Travel      1.009211                    0   
11     Boston Hotels Assoc      0.375120                    0   
12          Denver Tourism      0.846154                    0   
13   Atlanta Hotel Network      2.000000                    0   
14       Napa Valley Boar

In [27]:
def assign_risk(row):
    
    # --- Guardrails (avoid false positives) ---
    if (
        row["learning_phase"] == 1 and
        row["days_elapsed"] < 7
    ):
        return "Low"
    
    # --- Structural High Risk ---
    if (
        (row["pacing_ratio"] < 0.75 and row["days_remaining"] < 20) or
        (row["required_vs_actual_ratio"] > 2.2) or
        (row["tracking_issue"] == 1) or
        (row["legal_delay"] == 1 and row["days_remaining"] < 15)
    ):
        return "High"
    
    # --- Performance Degradation Risk ---
    if (
        (row["ctr_trend"] == "down" and row["pacing_ratio"] < 0.9) or
        (row["creative_issue"] == 1) or
        (row["bid_issue"] == 1 and row["pacing_ratio"] < 0.9)
    ):
        return "Medium"
    
    # --- Mild Under-pacing ---
    if row["pacing_ratio"] < 0.95:
        return "Medium"
    
    #--- elevate primium partner risk if under-pacing ---
    if row["premium_partner"] == 1 and row["pacing_ratio"] < 0.9:
        return "High"
    
    return "Low"

df["risk_level"] = df.apply(assign_risk, axis=1)

In [31]:
def compute_risk_score(row):
    
    # --- Normalize Components ---
    
    delivery_gap = max(0, min(1, 1 - row["pacing_ratio"]))
    
    recovery_pressure = max(
        0,
        min(1, (row["required_vs_actual_ratio"] - 1) / 2)
    )
    
    time_urgency = max(
        0,
        min(1, 1 - (row["days_remaining"] / row["days_total"]))
    )
    
    performance_risk = 1 if row["ctr_trend"] == "down" else 0
    
    blocker_score = min(
        row["creative_issue"] * 0.5 +
        row["tracking_issue"] * 1.0 +
        row["bid_issue"] * 0.5 +
        row["legal_delay"] * 0.5 +
        row["placement_issue"] * 0.5,
        1
    )
    
    # --- Weighted Sum (weights sum to 1) ---
    
    score = (
        0.30 * delivery_gap +
        0.25 * recovery_pressure +
        0.15 * time_urgency +
        0.10 * performance_risk +
        0.20 * blocker_score
    )
    
    # Premium boost (small adjustment, still bounded)
    if row["premium_partner"] == 1:
        score += 0.5
    
    # Learning phase dampener
    if row["learning_phase"] == 1 and row["days_elapsed"] < 7:
        score *= 0.7
    
    return round(score * 100, 1)

df["risk_score"] = df.apply(compute_risk_score, axis=1)

In [32]:
def map_risk_tier(score):
    if score >= 70:
        return "High"
    elif score >= 30:
        return "Medium"
    else:
        return "Low"

df["risk_level"] = df["risk_score"].apply(map_risk_tier)

In [33]:
print(df[["advertiser", "pacing_ratio", "premium_partner", "learning_phase", "risk_score", "risk_level"]])

                advertiser  pacing_ratio  premium_partner  learning_phase  \
0      Vegas Tourism Board      0.725000                1               0   
1          Orlando Tourism      0.555556                0               0   
2         NYC Hotels Group      0.961481                0               0   
3           Hawaii Resorts      1.437500                0               0   
4               DC Tourism      0.796190                0               0   
5       LA Boutique Hotels      0.720000                0               0   
6        Miami Beach Board      0.526531                0               0   
7      Chicago City Travel      0.972000                0               0   
8    Seattle Luxury Hotels      1.158730                0               0   
9          Phoenix Resorts      0.560000                0               0   
10        San Diego Travel      1.009211                1               0   
11     Boston Hotels Assoc      0.375120                0               0   

In [ ]:
def explain_risk(row):
    reasons = []

    # Pacing
    if row["pacing_ratio"] < 0.9:
        reasons.append(f"spend is behind schedule (pacing {row['pacing_ratio']:.2f} vs 1.00 target).")
    elif row["pacing_ratio"] > 1.1:
        reasons.append(f"spend is ahead of schedule (pacing {row['pacing_ratio']:.2f}).")

    # Required daily spend vs actual
    if row["required_vs_actual_ratio"] > 1.5:
        reasons.append("needs to increase daily spend materially to hit budget.")
    elif row["required_vs_actual_ratio"] < 0.7:
        reasons.append("can reduce daily spend and still hit budget.")

    # Time pressure
    if row["days_remaining"] < 14:
        reasons.append(f"only {row['days_remaining']} days left in flight.")
    elif row["learning_phase"] == 1 and row["days_elapsed"] < 7:
        reasons.append("early in learning phase; performance may still stabilize.")

    # Blockers from operational flags
    blockers = []
    if row.get("tracking_issue", 0) == 1:
        blockers.append("tracking issues")
    if row.get("creative_issue", 0) == 1:
        blockers.append("creative problems")
    if row.get("bid_issue", 0) == 1:
        blockers.append("bid constraints")
    if row.get("legal_delay", 0) == 1:
        blockers.append("legal approvals")
    if row.get("placement_issue", 0) == 1:
        blockers.append("inventory/placement issues")
    if blockers:
        reasons.append("blocked by " + ", ".join(blockers) + ".")

    # Performance trend
    if row.get("ctr_trend", "") == "down":
        reasons.append("CTR trending down vs prior period.")

    # Relationship importance
    if row.get("premium_partner", 0) == 1:
        reasons.append("premium partner; under-delivery carries higher relationship risk.")

    if not reasons:
        return "Campaign is pacing close to plan with no major blockers detected."

    return " ".join(reasons)


def suggest_action(row):
    # Priority 1: unblock hard issues
    if row.get("tracking_issue", 0) == 1:
        return "Work with ad ops to resolve tracking issues today, then re-validate delivery and performance."
    if row.get("legal_delay", 0) == 1:
        return "Escalate with legal and the client to clear approvals or extend flight dates to protect delivery."

    # Priority 2: severe under-delivery pressure
    if row["pacing_ratio"] < 0.8 and row["required_vs_actual_ratio"] > 1.5:
        return "Increase bids or loosen targeting, and add high-volume placements to accelerate spend over the next week."

    # Priority 3: moderate under-pacing
    if row["pacing_ratio"] < 0.95:
        if row.get("ctr_trend", "") == "down" or row.get("creative_issue", 0) == 1:
            return "Refresh creatives and review messaging/format; keep bids and budget stable while testing new variants."
        if row.get("bid_issue", 0) == 1:
            return "Raise bid caps or switch to a less restrictive bid strategy to unlock additional inventory."
        return "Broaden targeting or add more inventory sources while monitoring performance daily."

    # Priority 4: over-pacing / efficiency
    if row["pacing_ratio"] > 1.1:
        return "Tighten audience or placement targeting and consider lowering bids to preserve efficiency."

    # Priority 5: performance-only concerns
    if row.get("ctr_trend", "") == "down":
        return "Test new creatives and adjust placements focusing on higher-engagement surfaces."

    # Default: stable campaigns
    return "No immediate intervention needed; monitor pacing and performance in the next check-in."


# Attach explanations and recommended actions per campaign
df["risk_reason"] = df.apply(explain_risk, axis=1)
df["next_action"] = df.apply(suggest_action, axis=1)